In [ ]:
disable selinux
disable secure boot

Create new user on linux

In [ ]:
useradd admin

passwd admin

visudo

# wheel

sudo usermod -aG wheel admin

In [ ]:
# Open ssh for 
sudo bash -c 'echo "PermitRootLogin yes" > /etc/ssh/sshd_config.d/01-permitroot.conf'
sudo systemctl restart sshd

In [ ]:
sudo dnf install -y bash-completion

source /etc/profile.d/bash_completion.sh

sudo dnf install -y open-vm-tools
sudo systemctl enable --now vmtoolsd

sudo dnf install -y wget curl vim tar unzip tmux htop git tree

sudo dnf install -y net-tools iproute bind-utils telnet tcpdump traceroute mtr

sudo dnf install -y sysstat iotop lsof psmisc

sudo dnf install -y iptables ebtables socat conntrack ipset

sudo dnf update

#### For repo

In [ ]:
# Install required tools
sudo dnf install -y httpd createrepo_c dnf-utils

# Create directory for the mirror
sudo mkdir -p /var/www/html/centos-stream/9/x86_64

# Sync the packages that are already cached + new ones
sudo reposync --repoid=baseos      --download-metadata --newest-only -p /var/www/html/centos-stream/9/x86_64/
sudo reposync --repoid=appstream   --download-metadata --newest-only -p /var/www/html/centos-stream/9/x86_64/
sudo reposync --repoid=crb         --download-metadata --newest-only -p /var/www/html/centos-stream/9/x86_64/
sudo reposync --repoid=epel        --download-metadata --newest-only -p /var/www/html/centos-stream/9/x86_64/


In [ ]:
sudo systemctl enable --now httpd
sudo firewall-cmd --permanent --add-service=http
sudo firewall-cmd --reload

In [ ]:
sudo dnf clean all
sudo dnf makecache

In [ ]:
# Install web server
sudo dnf install -y httpd

# Make sure the directory structure is good
sudo mkdir -p /var/www/html/centos-stream/9/x86_64/{baseos,appstream,crb,epel}/Packages

# Fix permissions
sudo chown -R apache:apache /var/www/html/centos-stream
sudo chmod -R 755 /var/www/html/centos-stream

# Start and enable httpd
sudo systemctl enable --now httpd
sudo firewall-cmd --permanent --add-service=http
sudo firewall-cmd --reload

In [ ]:
# Remove bad package if it exists
sudo rm -f /var/www/html/centos-stream/9/x86_64/appstream/Packages/dotnet-sdk-*.rpm

# Recreate repodata
sudo createrepo_c --no-database /var/www/html/centos-stream/9/x86_64/appstream
sudo createrepo_c --no-database /var/www/html/centos-stream/9/x86_64/baseos

In [ ]:
sudo dnf config-manager --set-disabled baseos appstream crb epel epel-next epel-cisco-openh264 extras-common
sudo dnf clean all
sudo dnf repolist

On other VMs

In [ ]:
sudo dnf config-manager --set-disabled local-appstream
sudo dnf config-manager --set-disabled local-baseos

In [ ]:
sudo rm -f /etc/yum.repos.d/local-centos.repo

sudo tee /etc/yum.repos.d/local-centos.repo << 'EOF'
# === LOCAL REPOS - HIGHEST PRIORITY ===

[local-baseos]
name=Local CentOS Stream 9 - BaseOS
baseurl=http://172.16.6.66/centos-stream/9/x86_64/baseos
enabled=1
gpgcheck=0
priority=1

[local-appstream]
name=Local CentOS Stream 9 - AppStream
baseurl=http://172.16.6.66/centos-stream/9/x86_64/appstream
enabled=1
gpgcheck=0
priority=1

[local-crb]
name=Local CentOS Stream 9 - CRB
baseurl=http://172.16.6.66/centos-stream/9/x86_64/crb
enabled=0
gpgcheck=0
priority=1
EOF

In [ ]:
sudo dnf clean all
sudo dnf makecache --refresh
sudo dnf repolist
sudo dnf config-manager --set-enabled baseos appstream epel epel-next epel-cisco-openh264 extras-common
sudo dnf clean all
sudo dnf update

#### For HDD Storage problem ( bad sectors )

In [ ]:
sudo dnf install smartmontools -y
sudo smartctl -H /dev/sda

In [ ]:
sudo badblocks -v /dev/sda1 > /tmp/bad-sectors.txt
sudo e2fsck -l /tmp/bad-sectors.txt /dev/sda1